# iSWAP Module

The iSWAP is a plate transport gripper arm on the Hamilton STAR(let). After {meth}`~pylabrobot.hamilton.liquid_handlers.star.star._HamiltonSTAR.setup`, it is available as `star.iswap` — an {class}`~pylabrobot.capabilities.arms.orientable_arm.OrientableArm`.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pylabrobot.hamilton.liquid_handlers.star import STARLet

star = STARLet()
await star.setup()

2026-04-08 17:51:32,189 - pylabrobot.io.usb - INFO - Finding USB device...
2026-04-08 17:51:32,196 - pylabrobot.io.usb - INFO - Found USB device.
2026-04-08 17:51:32,197 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0


In [3]:
assert star.iswap is not None, "iSWAP is not available on this robot"

## Deck layout

In [4]:
from pylabrobot.resources import PLT_CAR_L5AC_A00, Cor_96_wellplate_360ul_Fb

plate_carrier = PLT_CAR_L5AC_A00(name="plate_carrier")
plate_carrier[1] = plate = Cor_96_wellplate_360ul_Fb(name="plate_01")
star.deck.assign_child_resource(plate_carrier, rails=15)

## Moving resources

### `move_resource`

The simplest way to move a plate between locations. This picks up the resource, moves it, and drops it in one call.

For fine-grained control over pickup and drop, pass `backend_params` with {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.PickUpParams` or {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.DropParams`.

In [5]:
from pylabrobot.hamilton.liquid_handlers.star.iswap import iSWAPBackend

await star.iswap.move_resource(plate, plate_carrier[2])

2026-04-08 17:51:45,926 - pylabrobot.hamilton.liquid_handlers.star.iswap - INFO - [iSWAP] pick up plate: x=482.9, y=210.2, z=192.3, direction=0 deg, width=127.8 mm
2026-04-08 17:51:57,374 - pylabrobot.hamilton.liquid_handlers.star.iswap - INFO - [iSWAP] release plate: x=482.9, y=306.2, z=192.3, direction=0 deg, width=127.8 mm


In [6]:
await star.iswap.move_resource(
  plate, plate_carrier[1],
  pickup_backend_params=iSWAPBackend.PickUpParams(grip_strength=8),
)

2026-04-08 17:52:02,893 - pylabrobot.hamilton.liquid_handlers.star.iswap - INFO - [iSWAP] pick up plate: x=482.9, y=306.2, z=192.3, direction=0 deg, width=127.8 mm
2026-04-08 17:52:07,809 - pylabrobot.hamilton.liquid_handlers.star.iswap - INFO - [iSWAP] release plate: x=482.9, y=210.2, z=192.3, direction=0 deg, width=127.8 mm


### Grip direction

By default, the iSWAP grips from the front. Use the `direction` parameter to change this.

This is the direction from which the iSWAP gripper will grip the plate. So if the direction is `LEFT`, the iSWAP will grip the plate from the left side (when looking at the front of the robot). The plate will be to the right of the iSWAP when gripping it.

In [7]:
from pylabrobot.capabilities.arms.standard import GripDirection

await star.iswap.pick_up_resource(plate, direction=GripDirection.LEFT)
await star.iswap.drop_resource(plate_carrier[2], direction=GripDirection.LEFT)

2026-04-08 17:52:13,325 - pylabrobot.hamilton.liquid_handlers.star.iswap - INFO - [iSWAP] pick up plate: x=482.9, y=210.2, z=192.3, direction=270 deg, width=85.5 mm
2026-04-08 17:52:21,538 - pylabrobot.hamilton.liquid_handlers.star.iswap - INFO - [iSWAP] release plate: x=482.9, y=306.2, z=192.3, direction=270 deg, width=85.5 mm


In [8]:
await star.iswap.move_resource(
  plate, plate_carrier[1],
  pickup_backend_params=iSWAPBackend.PickUpParams(grip_strength=8),
  pickup_direction=GripDirection.RIGHT,
  drop_direction=GripDirection.RIGHT,
)

2026-04-08 17:52:27,002 - pylabrobot.hamilton.liquid_handlers.star.iswap - INFO - [iSWAP] pick up plate: x=482.9, y=306.2, z=192.3, direction=90 deg, width=85.5 mm
2026-04-08 17:52:33,870 - pylabrobot.hamilton.liquid_handlers.star.iswap - INFO - [iSWAP] release plate: x=482.9, y=210.2, z=192.3, direction=90 deg, width=85.5 mm


## Common tasks

### Parking

Park the iSWAP. Pass {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.ParkParams` via `backend_params` to control the minimum traverse height.

In [9]:
await star.driver.iswap.park()

In [10]:
await star.driver.iswap.park(
  backend_params=iSWAPBackend.ParkParams(minimum_traverse_height=280),
)

### Manual movement

Move the iSWAP to an absolute or relative position. Useful for teaching and calibration. See [Manual movement (teaching / calibration)](#manual-movement-teaching--calibration) for a full walkthrough including coordinate conversion.

In [11]:
# absolute (mm)
await star.driver.iswap.move_x(200)

In [12]:
await star.driver.iswap.move_y(200)

In [13]:
await star.driver.iswap.move_z(280)

In [14]:
# relative (mm)
await star.driver.iswap.move_relative_x(step_size=10)
await star.driver.iswap.move_relative_y(step_size=10)
await star.driver.iswap.move_relative_z(step_size=-10)

### Opening the gripper

Open the iSWAP gripper using {meth}`~pylabrobot.capabilities.arms.orientable_arm.OrientableArm.open_gripper`. **Warning**: this will release any object that is gripped. Useful for error recovery.

In [15]:
await star.iswap.open_gripper(gripper_width=130)

### Closing the gripper

Close the iSWAP gripper. Pass {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.CloseGripperParams` via `backend_params` to control grip strength and plate width tolerance.

In [16]:
from pylabrobot.hamilton.liquid_handlers.star.errors import STARFirmwareError

try:
  await star.iswap.close_gripper(gripper_width=120)
except STARFirmwareError as e:
  print(f"As expected (no plate gripped): {e.errors}")

As expected (no plate gripped): {'ISWAP': NoElementError('Plate not found')}


In [17]:
try:
  await star.iswap.close_gripper(
    gripper_width=85,
    backend_params=iSWAPBackend.CloseGripperParams(grip_strength=8, plate_width_tolerance=1.0),
  )
except STARFirmwareError as e:
  print(f"As expected (no plate gripped): {e.errors}")

As expected (no plate gripped): {'ISWAP': NoElementError('Plate not found')}


## Rotations

```{warning}
Rotating the iSWAP can cause collisions with other components on the deck. 
Before rotating, move the iSWAP to a safe position clear of carriers and channels.
```

You can rotate the iSWAP to 12 predefined positions using 
{meth}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.rotate`.

![iSWAP positions](img/iswap_positions.png)

The `rotate` method takes a `rotation_drive` orientation and the final `grip_direction` 
of the iSWAP (with respect to the deck). The internal motion planner automatically adjusts the wrist drive.

In [18]:
await star.driver.iswap.move_x(200)
await star.driver.iswap.move_y(200)
await star.driver.iswap.move_z(280)

In [19]:
from pylabrobot.capabilities.arms.standard import GripDirection

await star.driver.iswap.rotate(
  rotation_drive=iSWAPBackend.RotationDriveOrientation.RIGHT,
  grip_direction=GripDirection.LEFT,
)

### Controlling the wrist and rotation drive individually

You can also control the wrist (T-drive) and rotation drive (W-drive) individually.

In [20]:
# Rotation drive: LEFT, RIGHT, or FRONT
# Wrist drive: LEFT, RIGHT, STRAIGHT, or REVERSE
await star.driver.iswap.rotate_rotation_drive(iSWAPBackend.RotationDriveOrientation.LEFT)
await star.driver.iswap.rotate_wrist(iSWAPBackend.WristDriveOrientation.REVERSE)

## Slow movement

Use the {meth}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.slow` context manager to reduce speed during sensitive operations. Pass {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.MoveToLocationParams` via `backend_params` for additional control over acceleration and collision avoidance.

In [21]:
async with star.driver.iswap.slow():
  await star.iswap.move_resource(plate, plate_carrier[2])
  await star.iswap.move_resource(plate, plate_carrier[1])

2026-04-08 17:53:07,102 - pylabrobot.hamilton.liquid_handlers.star.iswap - INFO - [iSWAP] pick up plate: x=482.9, y=210.2, z=192.3, direction=0 deg, width=127.8 mm
2026-04-08 17:53:17,699 - pylabrobot.hamilton.liquid_handlers.star.iswap - INFO - [iSWAP] release plate: x=482.9, y=306.2, z=192.3, direction=0 deg, width=127.8 mm
2026-04-08 17:53:23,193 - pylabrobot.hamilton.liquid_handlers.star.iswap - INFO - [iSWAP] pick up plate: x=482.9, y=306.2, z=192.3, direction=0 deg, width=127.8 mm
2026-04-08 17:53:28,047 - pylabrobot.hamilton.liquid_handlers.star.iswap - INFO - [iSWAP] release plate: x=482.9, y=210.2, z=192.3, direction=0 deg, width=127.8 mm


## Manual movement (teaching / calibration)

### 1. Clear the Y range

For safety, move the other components as far away as possible before teaching.

In [22]:
await star.driver.pip.position_components_for_free_iswap_y_range()

'C0FYid0072er00/00'

### 2. Set rotation

Move the iSWAP wrist and rotation drive to the correct orientation as [explained above](#rotations). Be careful to move the iSWAP to a position where it does not hit any other components.

### 3. Absolute movement

Use the following methods to move the iSWAP to absolute X, Y and Z positions. All units are in mm.

In [23]:
await star.driver.iswap.move_x(x=200)

In [24]:
await star.driver.iswap.move_y(y=200)

In [25]:
await star.driver.iswap.move_z(z=270)

### 4. Computing plate position from calibration

The x, y, and z values refer to the **center** of the iSWAP gripper, making them agnostic to plate size. In PLR, however, all locations are with respect to the left-front-bottom (LFB) of the plate. To convert from the calibrated center to LFB, subtract the plate's center-bottom anchor:

In [26]:
from pylabrobot.resources import Coordinate

x, y, z = 200, 200, 270  # calibrated center position
calibrated_position = Coordinate(x, y, z)
plate_lfb_absolute = calibrated_position - plate.get_anchor("c", "c", "b")

The plate's LFB is now in absolute coordinates. If the plate is a child of some parent resource, compute the relative location by subtracting the parent's absolute position:

In [27]:
parent = plate_carrier  # example: the plate's parent resource
parent_absolute = parent.get_location_wrt(star.deck)
plate_relative = plate_lfb_absolute - parent_absolute

Use this with `parent.assign_child_resource(plate, location=plate_relative)` to place the plate at the calibrated position.

### 5. Relative movements

You can also move the iSWAP relative to its current position. All units are in mm. These refer to the center of the gripper.

In [28]:
await star.driver.iswap.move_relative_x(step_size=10)

In [29]:
await star.driver.iswap.move_relative_y(step_size=10)

In [30]:
await star.driver.iswap.move_relative_z(step_size=10)

## Teardown

In [31]:
await star.stop()

2026-04-08 17:53:40,341 - pylabrobot.io.usb - WARNING - Closing connection to USB device.
